In [292]:
%pip install -q kagglehub

In [293]:
import kagglehub
import pandas as pd
from pathlib import Path

In [294]:
dataset= "danielsowah123/retail-sales-dataset"
path = kagglehub.dataset_download(dataset)

Using Colab cache for faster access to the 'retail-sales-dataset' dataset.


# Ventas globales y de Europa

##1.- Diseñar un pipeline el cual obtiene los registros de ventas de todas las regiones

In [295]:
ruta = Path(path)
archivos = list(ruta.glob("*.csv"))
df = pd.read_csv(archivos[0])
df.head()

,Transaction_ID,Order_ID,Customer_ID,Customer_Name,Customer_Age,Customer_Gender,Customer_Segment,Order_Date,Order_Time,Sales_Channel,...,Order_Status,Shipping_Method,Delivery_Days,Return_Flag,Return_Reason,Sales_Representative,Promotion_Code,Customer_Rating,Inventory_Level,Order_Year
0,T02016694,O0116694,C01482,Sophia Smith,28.0,Male,Consumer,2022-10-06,11:30:55,Online,...,Completed,Standard,4.0,No,NaN,James Anderson,SAVE10,4.0,81.0,2022
1,T02016374,O0108187,C01591,Hannah King,46.0,Female,Consumer,2024-06-03,17:06:29,Online,...,Completed,Standard,6.0,No,NaN,Ella Lewis,NaN,5.0,123.0,2024
2,T02004099,O0104099,C04140,Emma Wilson,42.0,Male,Consumer,2024-12-19,08:22:11,Online,...,Shipped,Standard,4.0,No,NaN,Liam Davis,NaN,4.0,93.0,2024
3,T02010280,O0110280,C03663,Daniel Clark,18.0,Male,Corporate,2025-10-05,14:32:43,B2B Portal,...,Completed,Next Day,2.0,No,NaN,Grace Harris,HOLIDAY20,NaN,59.0,2025
4,T02012685,O0112685,C02749,Sophia Smith,37.0,Male,Returning Customer,2022-09-13,14:55:16,B2B Portal,...,Completed,Express,2.0,No,NaN,Sophia Brown,NaN,2.0,56.0,2022


Exploración de registros

In [296]:
print("Numero de registros:", len(df))
print("Numero de columnas:", len(df.columns))

Numero de registros: 18045
Numero de columnas: 36


In [297]:
print(df.isnull().sum())

Transaction_ID              0
Order_ID                    0
Customer_ID                 0
Customer_Name             108
Customer_Age              218
Customer_Gender           162
Customer_Segment            0
Order_Date                  0
Order_Time                  0
Sales_Channel               0
Store_ID                    0
Store_Name                  0
Country                     0
Region                      0
City                        0
Product_ID                  0
Product_Name              144
Product_Category            0
Product_Subcategory         0
Quantity                    0
Unit_Price                  0
Discount_Percentage         0
Sales_Amount                0
Cost_Amount                 0
Profit                      0
Payment_Method            147
Order_Status                0
Shipping_Method             0
Delivery_Days             162
Return_Flag                 0
Return_Reason           16168
Sales_Representative        0
Promotion_Code           9737
Customer_R

Limpieza y transforrmación de datos

In [298]:
ventas = df.copy()
ventas.columns = (
    ventas.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
ventas = ventas.dropna(how="all")

for columna in ventas.select_dtypes(include="object").columns:
    ventas[columna] = ventas[columna].str.strip()

print("registros originales:", len(df))
print("registros procesados:", len(ventas))

display(ventas.head())

registros originales: 18045
registros procesados: 18045


,transaction_id,order_id,customer_id,customer_name,customer_age,customer_gender,customer_segment,order_date,order_time,sales_channel,...,order_status,shipping_method,delivery_days,return_flag,return_reason,sales_representative,promotion_code,customer_rating,inventory_level,order_year
0,T02016694,O0116694,C01482,Sophia Smith,28.0,Male,Consumer,2022-10-06,11:30:55,Online,...,Completed,Standard,4.0,No,NaN,James Anderson,SAVE10,4.0,81.0,2022
1,T02016374,O0108187,C01591,Hannah King,46.0,Female,Consumer,2024-06-03,17:06:29,Online,...,Completed,Standard,6.0,No,NaN,Ella Lewis,NaN,5.0,123.0,2024
2,T02004099,O0104099,C04140,Emma Wilson,42.0,Male,Consumer,2024-12-19,08:22:11,Online,...,Shipped,Standard,4.0,No,NaN,Liam Davis,NaN,4.0,93.0,2024
3,T02010280,O0110280,C03663,Daniel Clark,18.0,Male,Corporate,2025-10-05,14:32:43,B2B Portal,...,Completed,Next Day,2.0,No,NaN,Grace Harris,HOLIDAY20,NaN,59.0,2025
4,T02012685,O0112685,C02749,Sophia Smith,37.0,Male,Returning Customer,2022-09-13,14:55:16,B2B Portal,...,Completed,Express,2.0,No,NaN,Sophia Brown,NaN,2.0,56.0,2022


In [299]:
columnas_geo= [
    col for col in ventas.columns
    if any(
        palabra in col
        for palabra in ["region", "country", "pais", "location"]
    )
]

print("Columnas geograficas:", columnas_geo)

if columnas_geo:
    for col in columnas_geo:
        print(f"\nValores encontrados de {col}:")
        print(ventas[col].value_counts(dropna=False))
else:
    print("no existen columnas geograficas")

Columnas geograficas: ['country', 'region']

Valores encontrados de country:
country
United States     7544
United Kingdom    2981
Canada            2685
Germany           2132
France            1586
Australia         1117
Name: count, dtype: int64

Valores encontrados de region:
region
England                   2981
Northeast                 2159
West                      2149
South                     1662
Île-de-France             1586
Ontario                   1566
Midwest                   1558
Bavaria                   1226
New South Wales           1117
British Columbia          1113
North Rhine-Westphalia     902
North East                  16
British columbia             6
NRW                          4
Name: count, dtype: int64


NOTA: Se observa que habia nombres inconsistentes, como "British Columbia" con "British columbia", y "Northeast" con "North East".También de "NRW"= "North Rhine-Westphalia". Es por eso que se estandarizan con variantes que ya aparecen, para que no se ontrepeten más adelante como regiones diferentes

In [300]:
ventas["country"] = ventas["country"].str.strip()
ventas["region"] = ventas["region"].str.strip()

# Estandarizar variantes conocidas
ventas["region"] = ventas["region"].replace({
    "British columbia": "British Columbia",
    "NRW": "North Rhine-Westphalia",
    "North Rhine-westphalia": "North Rhine-Westphalia",
    "Northeast": "North East"
})

Se puede observar que ya no se interpretan por separado como anteriormente

In [301]:
columnas_geo= [
    col for col in ventas.columns
    if any(
        palabra in col
        for palabra in ["region", "country", "pais", "location"]
    )
]

print("Columnas geograficas:", columnas_geo)

if columnas_geo:
    for col in columnas_geo:
        print(f"\nValores encontrados de {col}:")
        print(ventas[col].value_counts(dropna=False))
else:
    print("no existen columnas geograficas")

Columnas geograficas: ['country', 'region']

Valores encontrados de country:
country
United States     7544
United Kingdom    2981
Canada            2685
Germany           2132
France            1586
Australia         1117
Name: count, dtype: int64

Valores encontrados de region:
region
England                   2981
North East                2175
West                      2149
South                     1662
Île-de-France             1586
Ontario                   1566
Midwest                   1558
Bavaria                   1226
British Columbia          1119
New South Wales           1117
North Rhine-Westphalia     906
Name: count, dtype: int64


comprobación

In [302]:
variantes = [
    "British columbia",
    "NRW",
    "North Rhine-westphalia",
    "Northeast"
]

print(
    ventas["region"]
    .isin(variantes)
    .sum()
)

0


Consulta en SQL

In [303]:
!apt-get update -qq
!apt-get install -y -qq postgresql > /dev/null
!service postgresql start


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ]


In [304]:
!sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='root'" | grep -q 1 || sudo -u postgres createuser root

!sudo -u postgres psql -tAc "SELECT 1 FROM pg_database WHERE datname='retaildb'" | grep -q 1 || sudo -u postgres createdb -O root retaildb

Validación de datos

In [305]:
%pip install -q sqlalchemy "psycopg[binary]"

from sqlalchemy import create_engine

engine = create_engine(
    "postgresql+psycopg://root@/retaildb"
    "?host=/var/run/postgresql"
)

almacenamiento de registros de ventas

In [306]:
ventas.to_sql(
    "ventas_globales",
    engine,
    if_exists="replace",
    index=False
)

print("daatos almacenados")

daatos almacenados


Consulta de registros

In [307]:
consulta = """
SELECT
    country,
    region,
    COUNT(*) AS total_registros
FROM ventas_globales
GROUP BY country, region
ORDER BY total_registros DESC;
"""

resumen_global = pd.read_sql(consulta, engine)

display(resumen_global)

,country,region,total_registros
0,United Kingdom,England,2981
1,United States,North East,2175
2,United States,West,2149
3,United States,South,1662
4,France,Île-de-France,1586
5,Canada,Ontario,1566
6,United States,Midwest,1558
7,Germany,Bavaria,1226
8,Canada,British Columbia,1119
9,Australia,New South Wales,1117


In [308]:
consulta = """
SELECT *
FROM ventas_globales;
"""

ventas_globales = pd.read_sql(consulta, engine)

print("total de registros:", len(ventas_globales))
display(ventas_globales.head())

total de registros: 18045


,transaction_id,order_id,customer_id,customer_name,customer_age,customer_gender,customer_segment,order_date,order_time,sales_channel,...,order_status,shipping_method,delivery_days,return_flag,return_reason,sales_representative,promotion_code,customer_rating,inventory_level,order_year
0,T02016694,O0116694,C01482,Sophia Smith,28.0,Male,Consumer,2022-10-06,11:30:55,Online,...,Completed,Standard,4.0,No,None,James Anderson,SAVE10,4.0,81.0,2022
1,T02016374,O0108187,C01591,Hannah King,46.0,Female,Consumer,2024-06-03,17:06:29,Online,...,Completed,Standard,6.0,No,None,Ella Lewis,None,5.0,123.0,2024
2,T02004099,O0104099,C04140,Emma Wilson,42.0,Male,Consumer,2024-12-19,08:22:11,Online,...,Shipped,Standard,4.0,No,None,Liam Davis,None,4.0,93.0,2024
3,T02010280,O0110280,C03663,Daniel Clark,18.0,Male,Corporate,2025-10-05,14:32:43,B2B Portal,...,Completed,Next Day,2.0,No,None,Grace Harris,HOLIDAY20,NaN,59.0,2025
4,T02012685,O0112685,C02749,Sophia Smith,37.0,Male,Returning Customer,2022-09-13,14:55:16,B2B Portal,...,Completed,Express,2.0,No,None,Sophia Brown,None,2.0,56.0,2022


Validacion

In [309]:
total_original = len(df)
total_postgresql = len(ventas_globales)
total_regiones = resumen_global["total_registros"].sum()

print("Registros originales:", total_original)
print("Registros en PostgreSQL:", total_postgresql)
print("Registros agrupados:", total_regiones)

Registros originales: 18045
Registros en PostgreSQL: 18045
Registros agrupados: 18045


Resultados del punto 1:

Implementamos un pipeline para obtener los registros de ventas del dataset.
También se utilizó SQL para la recuperacion almacenada y se realizo una agrupación por país y región, esto para distribuir las transacciones de manera geográfica.
Al final se verificó la cantidad de registros, para poder comprobar el proceso.

## 2.- Clasificar la región de europea y obtener su total de ventas de la región


Del analisis del punto 1, hemos identificadio los países europeos, los cuales son "United Kingdom", "Germany" y "France".

In [310]:
consulta = """
SELECT
    CASE
        WHEN country IN ('United Kingdom', 'Germany', 'France')
        THEN 'Europa'
        ELSE 'Otras regiones'
    END AS continente,
    COUNT(*) AS total_registros,
    SUM(sales_amount) AS ventas_totales
FROM ventas_globales
GROUP BY continente
ORDER BY ventas_totales DESC;
"""
clasificacion = pd.read_sql(consulta, engine)
display(clasificacion)

,continente,total_registros,ventas_totales
0,Otras regiones,11346,5227775.77
1,Europa,6699,3024365.63


In [311]:
consulta = """
SELECT
    country,
    region,
    COUNT(*) AS total_registros,
    SUM(sales_amount) AS ventas_totales
FROM ventas_globales
WHERE country IN ('United Kingdom', 'Germany', 'France')
GROUP BY country, region
ORDER BY ventas_totales DESC;
"""

ventas_europa = pd.read_sql(consulta, engine)

display(ventas_europa)

,country,region,total_registros,ventas_totales
0,United Kingdom,England,2981,1330780.58
1,France,Île-de-France,1586,727619.06
2,Germany,Bavaria,1226,584156.38
3,Germany,North Rhine-Westphalia,906,381809.61


Total de ventas de Europa

In [312]:
consulta = """
SELECT
    COUNT(*) AS total_registros,
    SUM(sales_amount) AS ventas_totales_europa
FROM ventas_globales
WHERE country IN ('United Kingdom', 'Germany', 'France');
"""

total_europa = pd.read_sql(consulta, engine)

display(total_europa)

,total_registros,ventas_totales_europa
0,6699,3024365.63


Validación

In [313]:
suma_regiones = ventas_europa["ventas_totales"].sum()
total_general = total_europa["ventas_totales_europa"].iloc[0]

print(f"ventas regiones: {suma_regiones:,.2f}")
print(f"Ventas Europa: {total_general:,.2f}")

assert abs(suma_regiones - total_general) < 0.01

assert (
    ventas_europa["total_registros"].sum()
    == total_europa["total_registros"].iloc[0]
)

ventas regiones: 3,024,365.63
Ventas Europa: 3,024,365.63


Resultados del punto 2

Consultamos con SQL para identificar las transacciones de países europeos presentes en el data, como REino Unido, Alemania y Francia.Finalmente se validó la información para ver que coincidieran.

## 3.- Desarrollar o diseñar en total de ventas por cada país de europa


Del punto 2 hemos obtenido que las ventas totales de Europa son 3,024,365.63, distribuidas en 6,699 registros.

In [314]:
consulta = """
SELECT
    country AS pais,
    COUNT(*) AS total_registros,
    SUM(sales_amount) AS ventas_totales
FROM ventas_globales
WHERE country IN ('United Kingdom', 'Germany', 'France')
GROUP BY country
ORDER BY ventas_totales DESC;
"""

ventas_paises = pd.read_sql(consulta, engine)

display(ventas_paises)

,pais,total_registros,ventas_totales
0,United Kingdom,2981,1330780.58
1,Germany,2132,965965.99
2,France,1586,727619.06


Creamos una columna que indique el porcentaje que representa cada país al total europeo, la cual se calcula facilmente

$$\text{Participacion} =\dfrac{\text{Ventas del pais i}}{\text{Ventas totales de Europa}} \cdot 100$$

In [315]:
ventas_paises["participacion_europa"] = (
    ventas_paises["ventas_totales"]
    / ventas_paises["ventas_totales"].sum()
    * 100
)

display(ventas_paises.round(2))

,pais,total_registros,ventas_totales,participacion_europa
0,United Kingdom,2981,1330780.58,44.00
1,Germany,2132,965965.99,31.94
2,France,1586,727619.06,24.06


validación de resultados

In [316]:
total_paises = ventas_paises ["ventas_totales"].sum()

consulta = """
SELECT SUM(sales_amount) AS ventas_totales_europa
FROM ventas_globales
WHERE country IN ('United Kingdom', 'Germany', 'France');
"""
resultado = pd.read_sql(consulta, engine)
total_europa_valor = resultado["ventas_totales_europa"].iloc[0]

print(f"Ventas Europa: {total_europa_valor:,.2f}")
print(f"Ventas paises: {total_paises:,.2f}")

assert abs(total_europa_valor - total_paises) < 0.01
print("validacion correcta")

Ventas Europa: 3,024,365.63
Ventas paises: 3,024,365.63
validacion correcta


In [317]:
ventas_paises.to_csv(
    "/content/ventas_paises_europa.csv",
    index=False,
    encoding="utf-8-sig"
)
# descarga el archvio csv
# from google.colab import files

# files.download("/content/ventas_paises_europa.csv")

Resultados del punto 3

Se desarrollo una consulta para obtener el total de ventas de cada país europeo. Los resultados fueron agrupadoss por país y ordenados de mayor a menor de acuerdo a las ventas, además de que se calculo el porcentaje de participación del país.

## 4.- Reporte de % de ventas globales con indicadores de cuál región tiene bajo rendimiento.


In [318]:
consulta = """
SELECT
    COUNT(*) AS total_registros,
    SUM(sales_amount) AS ventas_globales
FROM ventas_globales;
"""
total_global = pd.read_sql(consulta, engine)

display(total_global)

,total_registros,ventas_globales
0,18045,8252141.4


realizamos una consulta respecto a los datos

In [319]:
consulta = """
SELECT
    country,
    region,
    COUNT(*) AS total_registros,
    SUM(sales_amount) AS ventas_totales,
    ROUND(
        100.0 * SUM(sales_amount)::numeric /
        NULLIF(SUM(SUM(sales_amount)::numeric) OVER(), 0),
        2
    ) AS porcentaje_global
FROM ventas_globales
GROUP BY country, region
ORDER BY porcentaje_global DESC;
"""

reporte_global = pd.read_sql(consulta, engine)
total_ventas = reporte_global["ventas_totales"].sum()

reporte_global["porcentaje_global"] = (
    reporte_global["ventas_totales"]
    / total_ventas
    * 100
)
display(reporte_global.round(2))

,country,region,total_registros,ventas_totales,porcentaje_global
0,United Kingdom,England,2981,1330780.58,16.13
1,United States,North East,2175,1013854.27,12.29
2,United States,West,2149,992442.67,12.03
3,United States,South,1662,762091.47,9.24
4,United States,Midwest,1558,742351.80,9.00
5,France,Île-de-France,1586,727619.06,8.82
6,Canada,Ontario,1566,686905.97,8.32
7,Germany,Bavaria,1226,584156.38,7.08
8,Canada,British Columbia,1119,516824.99,6.26
9,Australia,New South Wales,1117,513304.60,6.22


In [320]:
promedio = reporte_global["ventas_totales"].mean()

print(f"Promedio de ventas por region: {promedio:,.2f}")

Promedio de ventas por region: 750,194.67


In [321]:
umbral = promedio * 0.5

reporte_global["indicador"] = reporte_global[
    "ventas_totales"
].apply(
    lambda x: "bajo"
    if x < umbral
    else "alto"
)

display(reporte_global)

,country,region,total_registros,ventas_totales,porcentaje_global,indicador
0,United Kingdom,England,2981,1330780.58,16.126488,alto
1,United States,North East,2175,1013854.27,12.285954,alto
2,United States,West,2149,992442.67,12.026486,alto
3,United States,South,1662,762091.47,9.235075,alto
4,United States,Midwest,1558,742351.80,8.995869,alto
5,France,Île-de-France,1586,727619.06,8.817336,alto
6,Canada,Ontario,1566,686905.97,8.323972,alto
7,Germany,Bavaria,1226,584156.38,7.078846,alto
8,Canada,British Columbia,1119,516824.99,6.262920,alto
9,Australia,New South Wales,1117,513304.60,6.220259,alto


In [322]:
bajo_rendimiento = reporte_global[
    reporte_global["indicador"] == "bajo"
]

display(bajo_rendimiento)

,country,region,total_registros,ventas_totales,porcentaje_global,indicador


Validación

In [323]:
total_regiones = reporte_global["ventas_totales"].sum()
total_ventas = total_global["ventas_globales"].iloc[0]

print(f"Ventas globales: {total_ventas:,.2f}")
print(f"Suma de regiones: {total_regiones:,.2f}")

assert abs(total_regiones - total_ventas) < 0.01

assert (
    reporte_global["total_registros"].sum()
    == total_global["total_registros"].iloc[0]
)

Ventas globales: 8,252,141.40
Suma de regiones: 8,252,141.40


Exporta el reporte

In [324]:
reporte_global.to_csv(
    "/content/reporte_ventas_globales.csv",
    index=False,
    encoding="utf-8-sig"
)

# from google.colab import files

# files.download("/content/reporte_ventas_globales.csv")

Resultados del punto 4

Se desarrollo un reporte para analizar la participación de cada región en las ventas globales, además de un indicador de las regiones que se encuentran por debajo del 50% del promedio general.
Al final, utilizamos validaciones para verificar el importe total de ventas globales.
